In [16]:
def flatten_transcript(data):
    if isinstance(data["transcript"], list):
        return " ".join([turn["content"] for turn in data["transcript"]])
    return data["transcript"]

In [17]:
records = []
for path in data_dir.glob("*.json"):
    with open(path) as f:
        data = json.load(f)
    if "transcript" in data and "return_pct" in data:
        data["transcript"] = flatten_transcript(data)
        records.append(data)

print(f"Loaded {len(records)} transcripts")
print(f"Example return: {records[0]['return_pct']}")
print(f"Transcript preview: {records[0]['transcript'][:200]}")

Loaded 25 transcripts
Example return: 0.7885884177936406
Transcript preview: Good morning, everyone. Welcome to JPMorgan Chase's First Quarter 2024 Earnings Call. This call is being recorded. We will now go live to the presentation. Please hold on. Thank you very much, and goo


In [18]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")

# Test tokenization on one transcript
sample = records[0]["transcript"]
tokens = tokenizer(sample, max_length=512, truncation=True, padding="max_length", return_tensors="pt")

print(f"input_ids shape: {tokens['input_ids'].shape}")
print(f"attention_mask shape: {tokens['attention_mask'].shape}")
print(f"label: {records[0]['return_pct']}")

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

input_ids shape: torch.Size([1, 512])
attention_mask shape: torch.Size([1, 512])
label: 0.7885884177936406


In [19]:
import torch
from torch.utils.data import Dataset, DataLoader

class EarningsDataset(Dataset):
    def __init__(self, records, tokenizer, max_length=512):
        self.records = records
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.records)
    
    def __getitem__(self, idx):
        record = self.records[idx]
        encoding = self.tokenizer(
            record["transcript"],
            max_length=self.max_length,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label": torch.tensor(record["return_pct"], dtype=torch.float)
        }

dataset = EarningsDataset(records, tokenizer)
print(f"Dataset size: {len(dataset)}")
sample = dataset[0]
print(f"input_ids: {sample['input_ids'].shape}")
print(f"label: {sample['label']}")

Dataset size: 25
input_ids: torch.Size([512])
label: 0.7885884046554565


In [20]:
from transformers import AutoModel
import torch.nn as nn

class EarningsModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = AutoModel.from_pretrained("ProsusAI/finbert")
        self.dropout = nn.Dropout(0.1)
        self.regressor = nn.Linear(768, 1)
    
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = outputs.last_hidden_state[:, 0, :]
        cls = self.dropout(cls)
        return self.regressor(cls).squeeze(-1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = EarningsModel().to(device)
print(f"Model on: {device}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

/packages/miniconda-t2/20230523/envs/jupyter-cuda121-20230610/lib/python3.8/site-packages/torchvision/datapoints/__init__.py:12: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you may have in this issue: https://github.com/pytorch/vision/issues/6753, and you can also check out https://github.com/pytorch/vision/issues/7319 to learn more about the APIs that we suspect might involve future changes. You can silence this warning by calling torchvision.disable_beta_transforms_warning().
  warnings.warn(_BETA_TRANSFORMS_WARNING)
/packages/miniconda-t2/20230523/envs/jupyter-cuda121-20230610/lib/python3.8/site-packages/torchvision/transforms/v2/__init__.py:54: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may sti

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

/packages/miniconda-t2/20230523/envs/jupyter-cuda121-20230610/lib/python3.8/site-packages/torch/_utils.py:803: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


Model on: cuda
Parameters: 109,483,009


model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

In [21]:
from torch.utils.data import random_split

# Split 80/20 train/test
n_test = max(1, int(len(dataset) * 0.2))
n_train = len(dataset) - n_test
train_ds, test_ds = random_split(dataset, [n_train, n_test])

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=8)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
criterion = nn.MSELoss()

print(f"Train: {n_train} | Test: {n_test}")

# Training loop
for epoch in range(5):
    model.train()
    total_loss = 0
    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)
        
        optimizer.zero_grad()
        preds = model(input_ids, attention_mask)
        loss = criterion(preds, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    # Eval
    model.eval()
    val_loss = 0
    correct = 0
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)
            preds = model(input_ids, attention_mask)
            val_loss += criterion(preds, labels).item()
            correct += ((preds >= 0) == (labels >= 0)).sum().item()
    
    rmse = (val_loss / len(test_loader)) ** 0.5
    dir_acc = correct / n_test
    print(f"Epoch {epoch+1}/5 | Train Loss: {total_loss/len(train_loader):.4f} | Val RMSE: {rmse:.4f} | Dir Acc: {dir_acc:.2f}")

Train: 20 | Test: 5
Epoch 1/5 | Train Loss: 20.2071 | Val RMSE: 5.1745 | Dir Acc: 1.00
Epoch 2/5 | Train Loss: 13.5480 | Val RMSE: 4.3661 | Dir Acc: 1.00
Epoch 3/5 | Train Loss: 17.3249 | Val RMSE: 4.2113 | Dir Acc: 1.00
Epoch 4/5 | Train Loss: 16.8262 | Val RMSE: 4.3768 | Dir Acc: 1.00
Epoch 5/5 | Train Loss: 11.6616 | Val RMSE: 4.6740 | Dir Acc: 1.00


In [22]:
torch.save(model.state_dict(), "finbert_earnings.pt")
print("Model saved!")

Model saved!
